This notebook contains code on collecting sidewalk data from OSM for the generated grids and then calculating length of sidewalks for each 100m x 100m grid

In [1]:
import osmium
import shapely.geometry
import shapely.wkb
import geopandas as gpd
from shapely.geometry import box
import os
import subprocess
import glob
import gc
import sys
from multiprocess import Pool

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

In [2]:
# set path for grid files
# you might have grids generated here from the 'generate_grids.ipynb'
grid_path = 'data/*.parquet'

# create a list of all parquet grid files from the specified directory
grids_list = [parquet for parquet in glob.glob(grid_path)]
print(grids_list)

# # configure logging (recommended if you monitor processing over a lot of files)
# log_path = 'logs/street_walk.log'

# # ensure log directory exists
# log_dir = os.path.dirname(log_path)
# if not os.path.exists(log_dir):
#     os.makedirs(log_dir)
    
# logging.basicConfig(filename=log_path, level=logging.INFO,
#                     format='%(asctime)s:%(levelname)s:%(message)s', force=True)

['data/grids_100_745.parquet', 'data/grids_100_648.parquet', 'data/grids_100_810.parquet', 'data/grids_100_813.parquet', 'data/grids_100_744.parquet', 'data/grids_100_741.parquet', 'data/grids_100_743.parquet', 'data/grids_100_684.parquet', 'data/grids_100_765.parquet', 'data/grids_100_713.parquet', 'data/grids_100_812.parquet', 'data/grids_100_646.parquet', 'data/grids_100_719.parquet', 'data/grids_100_787.parquet', 'data/grids_100_569.parquet', 'data/grids_100_530.parquet', 'data/grids_100_869.parquet', 'data/grids_100_718.parquet', 'data/grids_100_685.parquet', 'data/grids_100_868.parquet', 'data/grids_100_607.parquet', 'data/grids_100_834.parquet', 'data/grids_100_832.parquet', 'data/grids_100_717.parquet', 'data/grids_100_529.parquet', 'data/grids_100_790.parquet', 'data/grids_100_811.parquet', 'data/grids_100_789.parquet', 'data/grids_100_604.parquet', 'data/grids_100_647.parquet', 'data/grids_100_830.parquet', 'data/grids_100_567.parquet', 'data/grids_100_742.parquet', 'data/gri

In [3]:
def is_walkable(tags):
    excluded_highways = {
        "abandoned", "bus_guideway", "construction", "cycleway", "motor",
        "no", "planned", "platform", "proposed", "raceway", "razed"
    }
    hwy = tags.get("highway")
    if not hwy or hwy in excluded_highways:
        return False
    if tags.get("foot") == "no":
        return False
    if tags.get("service") == "private":
        return False
    if tags.get("area") == "yes":
        return False
    return True


def get_relevant_tags(tags):
        return {
            k: tags.get(k)
            for k in ("highway", "foot", "sidewalk", "surface", "name")
            if tags.get(k) is not None
        }

    
def extract_walking_ways(pbf_path):
    wkbfab = osmium.geom.WKBFactory()
    geometries = []

    class Handler(osmium.SimpleHandler):
        def way(self, w):
            if is_walkable(w.tags):
                try:
                    wkb = wkbfab.create_linestring(w)
                    geom = shapely.wkb.loads(wkb, hex=True)
                    tag_subset = get_relevant_tags(w.tags)
                    geometries.append((w.id, geom, tag_subset))
                except:
                    pass  # skip malformed geometries

    handler = Handler()
    handler.apply_file(pbf_path, locations=True)

    if not geometries:
        return gpd.GeoDataFrame(columns=["id", "tags", "geometry"])

    ids, geoms, tags = zip(*geometries)
    return gpd.GeoDataFrame({
        "id": ids,
        "tags": tags,
        "geometry": geoms
    }, crs="EPSG:4326")

In [4]:
def calculate_sidewalk_length(group):
    """
    Calculates the total length of sidewalks within each grid.
    
    Args:
    - group (DataFrame): GeoDataFrame group containing sidewalk line strings.

    Returns:
    - float: Total length of sidewalks in the grid.
    """
    if not group.empty:
        union = group['retain_linestring'].union_all()
        if union:
            intersection_union = group.iloc[0]['geometry'].intersection(union)
            if intersection_union:
                return intersection_union.length
    return 0


def process_grid(grid_path):
    """
    Processes each grid to calculate sidewalk lengths.
    
    Args:
    - grid_path (str): Path to the grid file.
    """
    try:
        grid_number = grid_path.split('_')[-1].split('.')[0]

        grid_gdf = gpd.read_parquet(grid_path)
        output_dir = os.path.join('/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/', 'street_walk_data')
        output_file = f'street_walk_{grid_number}.parquet'
        
        os.makedirs(output_dir, exist_ok=True)
        
        if os.path.exists(os.path.join(output_dir, output_file)) and 'street_walk_length' in grid_gdf.columns:
            return

        if 'index' not in grid_gdf.columns:
            grid_gdf.reset_index(inplace=True)
        
        if 'street_walk_length' in grid_gdf.columns:
            grid_gdf.drop(columns=['street_walk_length'], inplace=True)

    
        # convert grid to WGS84 (lat/lon)
        grid_gdf_4326 = grid_gdf.to_crs('EPSG:4326')
        print(f'Started processing grid {grid_number}')
    
        bounds = grid_gdf_4326.total_bounds
        buffered_box = box(*bounds).buffer(0.01)  # ~1 km buffer in degrees
    
        # clip .pbf file
        clipped_pbf_path = f'data/{grid_number}.pbf'
        os.makedirs(os.path.dirname(clipped_pbf_path), exist_ok=True)
    
        bbox_str = ','.join(map(str, buffered_box.bounds))
        subprocess.run([
            'osmium', 'extract',
            '-b', bbox_str,
            '/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/united-kingdom-261028.osm.pbf',
            '-o', clipped_pbf_path,
            '--overwrite'
        ], check=True)
        print(f'clipped PBF for {grid_number}')
        
        # process with pyosmium
        gdf_edges = extract_walking_ways(f'data/{grid_number}.pbf')
        
        gdf_edges.to_crs(grid_gdf.crs, inplace=True)
        gdf_edges['retain_linestring'] = gdf_edges.geometry
        
        gdf_edges.to_parquet(os.path.join(output_dir, output_file))
        # logging.info(f'Saved sidewalks data to {output_file}')
        print(f'Saved sidewalks data to {output_file}')

        joined = gpd.sjoin(grid_gdf, gdf_edges, how="left", predicate='intersects')
        grouped_gdf = joined.groupby('index')
        grid_gdf['street_walk_length'] = grouped_gdf.apply(calculate_sidewalk_length)

        grid_gdf.to_parquet(grid_path)
        # logging.info(f'Successfully processed grid {grid_path}')
        print(f'Successfully processed grid {grid_path}')
        
        del gdf_edges, grid_gdf
        gc.collect()
    except Exception as e:
        # logging.error(f'Error processing grid {grid_path}: {e}')
        print(f'Error processing grid {grid_path}: {e}')

In [5]:
# # sequential
# for elem in grids_list:
#     process_grid(elem)

# parallel processsing if you want to process a lot of files
num_processes = 2

with Pool(processes=num_processes) as pool:
    pool.map(process_grid, grids_list)

Started processing grid 609
clipped PBF for 609
Saved sidewalks data to street_walk_609.parquet


/tmp/ipykernel_1251940/2811891587.py:79: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grid_gdf['street_walk_length'] = grouped_gdf.apply(calculate_sidewalk_length)


Successfully processed grid data/grids_100_609.parquet


In [6]:
import geopandas as gpd

gdf = gpd.read_parquet("data/grids_100_608.parquet")
gdf.head()

,index,geometry,grid_100000_id,slope,population,dou,ent_5,ent_10,ent_15,green_area,pub_trans_count,street_walk_length
0,0,"POLYGON ((3600000 3200000, 3600100 3200000, 36...",608,1.058738,146.612640,30,-0.0,0.402929,1.038636,0.000000,1,665.144041
1,1,"POLYGON ((3600000 3200100, 3600100 3200100, 36...",608,1.476905,121.456001,30,-0.0,0.390230,1.072553,0.000000,2,615.778385
2,2,"POLYGON ((3600000 3200200, 3600100 3200200, 36...",608,1.962831,26.572197,30,-0.0,0.450721,1.094878,4095.433905,0,331.956444
3,3,"POLYGON ((3600000 3200300, 3600100 3200300, 36...",608,2.380930,0.343318,30,-0.0,0.561137,1.096680,10000.000000,0,282.424101
4,4,"POLYGON ((3600000 3200400, 3600100 3200400, 36...",608,2.580068,0.000000,30,-0.0,0.666473,1.102429,10000.000000,0,310.391664
